##### Ingest constructors.json file

##### Read the JSON file using the spark dataframe reader

In [0]:
dbutils.widgets.text("p_data_source", "")
v_data_source = dbutils.widgets.get("p_data_source")

In [0]:
dbutils.widgets.text("p_file_date", "2021-03-21")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
constructor_schema = "constructorId INT, constructorRef STRING, name STRING, nationality STRING, url STRING"


In [0]:
constructors_df = spark.read \
    .schema(constructor_schema) \
    .json(f"{raw_path}/{v_file_date}/constructors.json")

##### Drop unwanted columns from the dataframe

In [0]:
constructors_dropped_df = constructors_df.drop(col("url"))

##### Rename columns and add ingestion date

In [0]:
constructors_final_df = constructors_dropped_df \
    .withColumnRenamed("constructorId", "constructor_id") \
    .withColumnRenamed("constructorRef", "constructor_ref") \
    .withColumn("data_source", lit(v_data_source)) \
    .withColumn("file_date", lit(v_file_date))

In [0]:
constructors_final_df = add_ingestion_date(constructors_final_df)

##### Write output to parquet file

In [0]:
constructors_final_df.write.mode("overwrite").parquet(f"{processed_path}/constructors")

##### Check

In [0]:
spark.read.parquet(f"{processed_path}/constructors").display()

In [0]:
dbutils.notebook.exit("SUCCESS")